In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec


In [ ]:
import os
import pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
pc.create_index(
    name="profspot-csuf", dimension=1536, metric="cosine", spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

In [ ]:
import json
data = json.load(open("reviews.json"))
data['reviews']

In [ ]:
from openai import OpenAI
processed_data = []
client = OpenAI()
reviews = data['reviews']

for start in range(0, len(reviews), 100):
    batch = reviews[start:start + 100]
    response = client.embeddings.create(
        input=[r['review'] for r in batch],
        model="text-embedding-3-small"
    )
    for i, (review, item) in enumerate(zip(batch, response.data)):
        processed_data.append({
            "values": item.embedding,
            "id": f"{start + i}-{review['professor']}",
            "metadata": {
                "professor": review["professor"],
                "review": review["review"],
                "subject": review["subject"],
                "stars": review["stars"]
            }
        })


In [ ]:
processed_data[0]

In [ ]:
index = pc.Index('profspot-csuf')
for i in range(0, len(processed_data), 100):
    index.upsert(vectors=processed_data[i:i + 100], namespace="csuf")


In [ ]:
index.describe_index_stats()